# DS1/GSE103224 — post-QC Scrublet doublet diagnostic

This notebook reproduces the original post-QC Scrublet check for
DS1/GSE103224.

Important interpretation:

- Scrublet is run on the complete post-QC DS1 object, matching the original
  analysis notebook.
- The resulting `doublet_score` and `predicted_doublet` fields are retained
  as diagnostic metadata.
- Predicted doublets are **not removed**, so the analytical cell count
  remains 23,207.
- This notebook therefore documents a doublet-screening diagnostic rather
  than an additional dataset-filtering step.

Input:
`data/processed/DS1_GSE103224/GSE103224_postQC.h5ad`

Output:
`data/processed/DS1_GSE103224/GSE103224_postQC_scrublet_checked.h5ad`

In [ ]:
import inspect
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS1_GSE103224"
        / "GSE103224_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS1_GSE103224"
        / "GSE103224_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS1_GSE103224"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (23_207, 29_457)
EXPECTED_ORIGINAL_PREDICTED_DOUBLETS = 1

# These values are the standard defaults used by the original
# sc.pp.scrublet(adata) call. Supported arguments are detected at runtime
# to retain compatibility across nearby Scanpy versions.
REQUESTED_SCRUBLET_PARAMETERS = {
    "sim_doublet_ratio": 2.0,
    "expected_doublet_rate": 0.05,
    "n_prin_comps": 30,
    "random_state": 0,
}

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds1_quality_control.ipynb first or set "
        "GSE103224_POSTQC_H5AD."
    )

In [ ]:
def sparse_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_nonnegative_integer_counts(matrix, label: str):
    values = sparse_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} has no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample_id" not in adata.obs:
    raise KeyError("sample_id is missing from adata.obs.")
if "counts" not in adata.layers:
    raise KeyError("The post-QC object is missing layers['counts'].")

assert_raw_nonnegative_integer_counts(adata.X, "adata.X")
assert_raw_nonnegative_integer_counts(
    adata.layers["counts"], 'adata.layers["counts"]'
)

# Make the notebook safe to rerun on an in-memory object.
for column in ("doublet_score", "predicted_doublet"):
    if column in adata.obs:
        del adata.obs[column]

print(adata)

In [ ]:
scrublet_signature = inspect.signature(sc.pp.scrublet)
supported_parameters = set(scrublet_signature.parameters)

scrublet_parameters = {
    key: value
    for key, value in REQUESTED_SCRUBLET_PARAMETERS.items()
    if key in supported_parameters
}

omitted_parameters = sorted(
    set(REQUESTED_SCRUBLET_PARAMETERS) - set(scrublet_parameters)
)
if omitted_parameters:
    print(
        "Parameters not supported by this Scanpy version and therefore "
        "omitted:",
        omitted_parameters,
    )

print("Scrublet parameters used:", scrublet_parameters)

sc.pp.scrublet(
    adata,
    **scrublet_parameters,
)

required_columns = {"doublet_score", "predicted_doublet"}
missing = required_columns - set(adata.obs.columns)
if missing:
    raise RuntimeError(
        f"Scrublet completed without creating required columns: {missing}"
    )

adata.obs["predicted_doublet"] = (
    adata.obs["predicted_doublet"].astype(bool)
)

predicted_count = int(adata.obs["predicted_doublet"].sum())
print(adata.obs["predicted_doublet"].value_counts())
print(adata.obs["doublet_score"].describe())

if predicted_count != EXPECTED_ORIGINAL_PREDICTED_DOUBLETS:
    print(
        "WARNING: the predicted-doublet count differs from the original "
        f"run ({EXPECTED_ORIGINAL_PREDICTED_DOUBLETS}). Observed: "
        f"{predicted_count}. This may reflect a package-version difference."
    )

In [ ]:
per_sample_summary = (
    adata.obs.groupby("sample_id", observed=True)
    .agg(
        cells=("predicted_doublet", "size"),
        predicted_doublets=("predicted_doublet", "sum"),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)
per_sample_summary["predicted_doublet_fraction"] = (
    per_sample_summary["predicted_doublets"]
    / per_sample_summary["cells"]
)
per_sample_summary.to_csv(
    AUDIT_DIR / "GSE103224_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

score_table = adata.obs[
    ["sample_id", "doublet_score", "predicted_doublet"]
].copy()
score_table.to_csv(
    AUDIT_DIR / "GSE103224_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(
    adata.obs["doublet_score"].to_numpy(),
    bins=60,
    edgecolor="black",
)
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE103224 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE103224_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [ ]:
predicted_count = int(adata.obs["predicted_doublet"].sum())

doublet_summary = {
    "dataset_id": "DS1_GSE103224",
    "geo_accession": "GSE103224",
    "scope": "complete post-QC DS1 object",
    "method": "scanpy.pp.scrublet",
    "parameters_requested": REQUESTED_SCRUBLET_PARAMETERS,
    "parameters_used": scrublet_parameters,
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "predicted_doublets": predicted_count,
    "predicted_doublet_fraction": float(predicted_count / adata.n_obs),
    "doublets_removed": False,
    "output_cells": int(adata.n_obs),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

# A separate key avoids modifying Scanpy's own Scrublet metadata.
adata.uns["doublet_detection_provenance"] = {
    "scope": "post_QC_dataset_level_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed": "false",
    "predicted_doublets": predicted_count,
    "parameters_json": json.dumps(scrublet_parameters, sort_keys=True),
}

with open(
    AUDIT_DIR / "GSE103224_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)

In [ ]:
if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly changed the object shape: "
        f"{adata.shape}"
    )

temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in ("doublet_score", "predicted_doublet"):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

print("Saved and validated:", OUTPUT_H5AD)